**Census Bureau Data: Median Income by Household (Zipcode and Year)**

This notebook brings together data from the Census Bureau's ACS datasets. The objective is to put together a dataset that has the median income per household from 2019 to 2025 by zipcode and year. Then we will add this data to the existing dataset of CCAD/DCAD/TAD home appraisal values to complete our dataset before running the model.

I am hoping that including median income per household by zipcode will offer some perspective on the demand side of the equation and increase the accuracy of the model. The name of the exact report I used is MEDIAN HOUSEHOLD INCOME IN THE PAST 12 MONTHS (IN 20XX INFLATION-ADJUSTED DOLLARS), I filtered the report by tract (and then converted that to zipcode using the GeoID and HUD Crosswalk files) and included all zipcodes for the three counties whose housing data I'm using (Collin County, Dallas County, Tarrant County).

In [1]:
import requests
import pandas as pd
import re

def load_census_json(url: str) -> pd.DataFrame:
    """
    Fetch JSON data from the Census API, convert to a DataFrame,
    and automatically extract the year from the URL.
    """
    # Extract year from the URL
    match = re.search(r'/data/(20\d{2})/', url)
    if match:
        year = int(match.group(1))
    else:
        raise ValueError("Could not extract year from the URL.")
    
    # Request data
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    
    # Load into DataFrame
    df = pd.DataFrame(data[1:], columns=data[0])
    
    # Add the year column
    df["year"] = year
    
    return df

In [2]:
# To load the datasets and create the df including the year as a new column

df_2019 = load_census_json("https://api.census.gov/data/2019/acs/acs5?get=group(B19013)&ucgid=pseudo(310M500US19100$1400000)")
df_2020 = load_census_json("https://api.census.gov/data/2020/acs/acs5?get=group(B19013)&ucgid=pseudo(310M600US19100$1400000)")
df_2021 = load_census_json("https://api.census.gov/data/2021/acs/acs5?get=group(B19013)&ucgid=pseudo(310M600US19100$1400000)")
df_2022 = load_census_json("https://api.census.gov/data/2022/acs/acs5?get=group(B19013)&ucgid=pseudo(310M600US19100$1400000)")
df_2023 = load_census_json("https://api.census.gov/data/2023/acs/acs5?get=group(B19013)&ucgid=pseudo(310M700US19100$1400000)")

In [3]:
# To combine the dataframes into a single large dataframe

df_CB = pd.concat([df_2019, df_2020, df_2021, df_2022, df_2023], axis=0, ignore_index=True)

df_CB.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   B19013_001E   8128 non-null   object
 1   B19013_001EA  111 non-null    object
 2   B19013_001M   8128 non-null   object
 3   B19013_001MA  111 non-null    object
 4   GEO_ID        8128 non-null   object
 5   NAME          8128 non-null   object
 6   ucgid         8128 non-null   object
 7   year          8128 non-null   int64 
dtypes: int64(1), object(7)
memory usage: 508.1+ KB


In [4]:
df_CB.head()

,B19013_001E,B19013_001EA,B19013_001M,B19013_001MA,GEO_ID,NAME,ucgid,year
0,250001,"250,000+",-333333333,***,1400000US48113007604,"Census Tract 76.04, Dallas County, Texas",1400000US48113007604,2019
1,250001,"250,000+",-333333333,***,1400000US48113013300,"Census Tract 133, Dallas County, Texas",1400000US48113013300,2019
2,250001,"250,000+",-333333333,***,1400000US48113013500,"Census Tract 135, Dallas County, Texas",1400000US48113013500,2019
3,250001,"250,000+",-333333333,***,1400000US48113019301,"Census Tract 193.01, Dallas County, Texas",1400000US48113019301,2019
4,250001,"250,000+",-333333333,***,1400000US48113019501,"Census Tract 195.01, Dallas County, Texas",1400000US48113019501,2019


In [5]:
# To change the column names on the dataframe to be more human readable, I based the names on the actual table from the Census Bureau site. 

df_CB = df_CB.rename(columns={'B19013_001E':'Median_H_Inc','NAME':'TRACT'})

df_CB.head()

,Median_H_Inc,B19013_001EA,B19013_001M,B19013_001MA,GEO_ID,TRACT,ucgid,year
0,250001,"250,000+",-333333333,***,1400000US48113007604,"Census Tract 76.04, Dallas County, Texas",1400000US48113007604,2019
1,250001,"250,000+",-333333333,***,1400000US48113013300,"Census Tract 133, Dallas County, Texas",1400000US48113013300,2019
2,250001,"250,000+",-333333333,***,1400000US48113013500,"Census Tract 135, Dallas County, Texas",1400000US48113013500,2019
3,250001,"250,000+",-333333333,***,1400000US48113019301,"Census Tract 193.01, Dallas County, Texas",1400000US48113019301,2019
4,250001,"250,000+",-333333333,***,1400000US48113019501,"Census Tract 195.01, Dallas County, Texas",1400000US48113019501,2019


In [6]:
# To drop the columns we don't need (i.e. margin of error, columns where values=none, GEO_ID, etc.)

df_CB = df_CB.drop(columns=['B19013_001EA','B19013_001M','B19013_001MA','ucgid'])

df_CB.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Median_H_Inc  8128 non-null   object
 1   GEO_ID        8128 non-null   object
 2   TRACT         8128 non-null   object
 3   year          8128 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 254.1+ KB


In [7]:
# To re-order the columns to be more human readable

new_order = ['year','TRACT','GEO_ID','Median_H_Inc']

df_CB = df_CB[new_order]

df_CB.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   year          8128 non-null   int64 
 1   TRACT         8128 non-null   object
 2   GEO_ID        8128 non-null   object
 3   Median_H_Inc  8128 non-null   object
dtypes: int64(1), object(3)
memory usage: 254.1+ KB


In [8]:
# To drop all rows with useless values in them

df_CB = df_CB[~(df_CB['Median_H_Inc'] == '-666666666')]

In [9]:
# To have only the 11 digit TRACT_ID show in a separate column (we will use this to join with zipcodes later)

df_CB.loc[:, 'TRACT_ID'] = df_CB['GEO_ID'].str.extract(r'(48\d{9})', expand=False)

df_CB.head()

,year,TRACT,GEO_ID,Median_H_Inc,TRACT_ID
0,2019,"Census Tract 76.04, Dallas County, Texas",1400000US48113007604,250001,48113007604
1,2019,"Census Tract 133, Dallas County, Texas",1400000US48113013300,250001,48113013300
2,2019,"Census Tract 135, Dallas County, Texas",1400000US48113013500,250001,48113013500
3,2019,"Census Tract 193.01, Dallas County, Texas",1400000US48113019301,250001,48113019301
4,2019,"Census Tract 195.01, Dallas County, Texas",1400000US48113019501,250001,48113019501


In [10]:
# Extract tract number and county

df_CB[['TRACT', 'county']] = df_CB['TRACT'].str.extract(
    r'Census Tract\s+([\d\.]+)[,;]\s+([\w\s]+) County'
)
df_CB.head()

,year,TRACT,GEO_ID,Median_H_Inc,TRACT_ID,county
0,2019,76.04,1400000US48113007604,250001,48113007604,Dallas
1,2019,133,1400000US48113013300,250001,48113013300,Dallas
2,2019,135,1400000US48113013500,250001,48113013500,Dallas
3,2019,193.01,1400000US48113019301,250001,48113019301,Dallas
4,2019,195.01,1400000US48113019501,250001,48113019501,Dallas


In [11]:
df_CB.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8093 entries, 0 to 8127
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   year          8093 non-null   int64 
 1   TRACT         8093 non-null   object
 2   GEO_ID        8093 non-null   object
 3   Median_H_Inc  8093 non-null   object
 4   TRACT_ID      8093 non-null   object
 5   county        8093 non-null   object
dtypes: int64(1), object(5)
memory usage: 442.6+ KB


**As of the creation of the file (12/07/2025) the ACS 2024 data was only very recently released on 12/04/2025. The latest data shows the median and mean incomes by county but does not yet have those numbers available by zipcode. I've decided to create numbers for the values by zip code by using the df_2023 data by zipcode and multiplying the median values by the YoY % increase in the County level median income that *IS* provided by the Census. The data used and calculations are below.**

In [12]:
# To load the median income in 2023 and 2024 per the CB and calculate the YoY% change. 

df_23inc = load_census_json('https://api.census.gov/data/2023/acs/acs5?get=group(B19013)&ucgid=0500000US48085,0500000US48113,0500000US48439')
df_24inc = load_census_json('https://api.census.gov/data/2024/acs/acs1?get=group(B19013)&ucgid=0500000US48085,0500000US48113,0500000US48439')

df_23inc.head()

,B19013_001E,B19013_001EA,B19013_001M,B19013_001MA,GEO_ID,NAME,ucgid,year
0,117588,None,1528,None,0500000US48085,"Collin County, Texas",0500000US48085,2023
1,74149,None,771,None,0500000US48113,"Dallas County, Texas",0500000US48113,2023
2,81905,None,930,None,0500000US48439,"Tarrant County, Texas",0500000US48439,2023


In [13]:
df_24inc.head()

,B19013_001E,B19013_001EA,B19013_001M,B19013_001MA,GEO_ID,NAME,ucgid,year
0,124316,None,3398,None,0500000US48085,"Collin County, Texas",0500000US48085,2024
1,78932,None,2070,None,0500000US48113,"Dallas County, Texas",0500000US48113,2024
2,85197,None,2023,None,0500000US48439,"Tarrant County, Texas",0500000US48439,2024


In [14]:
# To rename the columns to be more human readable and drop unnecessary columns

df_23inc = df_23inc.drop(columns=['B19013_001EA','B19013_001M','B19013_001MA','GEO_ID','ucgid'])

df_24inc = df_24inc.drop(columns=['B19013_001EA','B19013_001M','B19013_001MA','GEO_ID','ucgid'])

df_23inc = df_23inc.rename(columns={'B19013_001E':'Median_H_Inc'})

df_24inc = df_24inc.rename(columns={'B19013_001E':'Median_H_Inc'})

df_23inc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Median_H_Inc  3 non-null      object
 1   NAME          3 non-null      object
 2   year          3 non-null      int64 
dtypes: int64(1), object(2)
memory usage: 204.0+ bytes


In [15]:
df_24inc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Median_H_Inc  3 non-null      object
 1   NAME          3 non-null      object
 2   year          3 non-null      int64 
dtypes: int64(1), object(2)
memory usage: 204.0+ bytes


In [16]:
# To change the data type of the Median Household Income column to int in both df's

df_23inc['Median_H_Inc'] = df_23inc['Median_H_Inc'].astype('int')
df_24inc['Median_H_Inc'] = df_24inc['Median_H_Inc'].astype('int')

In [17]:
# To calculate the YoY% change by county(NAME) from 2023 - 2024, I will use the %change to calculate household income values for 2024 and 2025 by ZCTA

CCAD_change_24 = ((df_24inc.loc[df_24inc['NAME'] == 'Collin County, Texas', 'Median_H_Inc'].iloc[0]) / (df_23inc.loc[df_23inc['NAME'] == 'Collin County, Texas', 'Median_H_Inc'].iloc[0]))

DCAD_change_24 = ((df_24inc.loc[df_24inc['NAME'] == 'Dallas County, Texas', 'Median_H_Inc'].iloc[0]) / (df_23inc.loc[df_23inc['NAME'] == 'Dallas County, Texas', 'Median_H_Inc'].iloc[0]))

TAD_change_24 = ((df_24inc.loc[df_24inc['NAME'] == 'Tarrant County, Texas', 'Median_H_Inc'].iloc[0]) / (df_23inc.loc[df_23inc['NAME'] == 'Tarrant County, Texas', 'Median_H_Inc'].iloc[0]))

print(f'Collin County Yoy% Change: {CCAD_change_24:.2%}')
print(f'Dallas County Yoy% Change: {DCAD_change_24:.2%}')
print(f'Tarrant County Yoy% Change: {TAD_change_24:.2%}')

Collin County Yoy% Change: 105.72%
Dallas County Yoy% Change: 106.45%
Tarrant County Yoy% Change: 104.02%


In [18]:
# To create the 2024 Median Household Income (MHI) dataset by using the 2023 MHI and changing the year to 2024

df_2024 = df_2023

# To update year column
df_2024['year'] = 2024

df_2024.head()

,B19013_001E,B19013_001EA,B19013_001M,B19013_001MA,GEO_ID,NAME,ucgid,year
0,-666666666,-,-222222222,**,1400000US48085030902,Census Tract 309.02; Collin County; Texas,1400000US48085030902,2024
1,250001,"250,000+",-333333333,***,1400000US48085031668,Census Tract 316.68; Collin County; Texas,1400000US48085031668,2024
2,250001,"250,000+",-333333333,***,1400000US48085031706,Census Tract 317.06; Collin County; Texas,1400000US48085031706,2024
3,250001,"250,000+",-333333333,***,1400000US48113007301,Census Tract 73.01; Dallas County; Texas,1400000US48113007301,2024
4,250001,"250,000+",-333333333,***,1400000US48113007604,Census Tract 76.04; Dallas County; Texas,1400000US48113007604,2024


In [19]:
# To drop the columns we don't need (i.e. margin of error, columns where values=none, GEO_ID, etc.)

df_2024 = df_2024.drop(columns=['B19013_001EA','B19013_001M','B19013_001MA','ucgid'])

# To change the column names on the dataframe to be more human readable, I based the names on the actual table from the Census Bureau site. 

df_2024 = df_2024.rename(columns={'B19013_001E':'Median_H_Inc','NAME':'TRACT'})

# To re-order the columns to be more human readable
new_order = ['year','TRACT','GEO_ID','Median_H_Inc']

df_2024 = df_2024[new_order]

df_2024 = df_2024[~(df_2024['Median_H_Inc'] == '-666666666')]

# To have only the 11 digit TRACT_ID show in a separate column (we will use this to join with zipcodes later)

df_2024.loc[:, 'TRACT_ID'] = df_2024['GEO_ID'].str.extract(r'(48\d{9})', expand=False)

# Extract tract number and county
df_2024[['TRACT', 'county']] = df_2024['TRACT'].str.extract(r'Census Tract\s+([\d\.]+)[,;]\s+([\w\s]+) County')

df_2024.head()

,year,TRACT,GEO_ID,Median_H_Inc,TRACT_ID,county
1,2024,316.68,1400000US48085031668,250001,48085031668,Collin
2,2024,317.06,1400000US48085031706,250001,48085031706,Collin
3,2024,73.01,1400000US48113007301,250001,48113007301,Dallas
4,2024,76.04,1400000US48113007604,250001,48113007604,Dallas
5,2024,80,1400000US48113008000,250001,48113008000,Dallas


In [20]:
# To change MHI column values to integer data type
df_2024['Median_H_Inc'] = df_2024['Median_H_Inc'].astype(int)

# To update Median_H_Inc by county YoY %change
df_2024.loc[df_2024['county'] == 'Collin', 'Median_H_Inc'] = (df_2024.loc[df_2024['county'] == 'Collin', 'Median_H_Inc'] * CCAD_change_24).round().astype(int)

df_2024.loc[df_2024['county'] == 'Dallas', 'Median_H_Inc'] = (df_2024.loc[df_2024['county'] == 'Dallas', 'Median_H_Inc'] * DCAD_change_24).round().astype(int)

df_2024.loc[df_2024['county'] == 'Tarrant', 'Median_H_Inc'] = (df_2024.loc[df_2024['county'] == 'Tarrant', 'Median_H_Inc'] * TAD_change_24).round().astype(int)

df_2024.head()

,year,TRACT,GEO_ID,Median_H_Inc,TRACT_ID,county
1,2024,316.68,1400000US48085031668,264305,48085031668,Collin
2,2024,317.06,1400000US48085031706,264305,48085031706,Collin
3,2024,73.01,1400000US48113007301,266127,48113007301,Dallas
4,2024,76.04,1400000US48113007604,266127,48113007604,Dallas
5,2024,80,1400000US48113008000,266127,48113008000,Dallas


**The Census MHI data caps income by 250K + 1 in the datasets, but I chose to include the yoy% increase for these values because I don't want the model to think that the higher range incomes are static YoY and have that potentially result in less accuracy on price predictions for home in certain higher income zipcodes.**

In [21]:
# To create the 2025 Median Household Income (MHI) dataset by using the 2024 MHI and making the same changes I made to calculate the 2024 values

# Because I don't have the YoY% change from 2024 to 2025 I am going to re-use the YoY% change from 2023 to 2024

df_2025 = df_2024

# To update year column
df_2025['year'] = 2025

# To update Median_H_Inc by county YoY %change
df_2025.loc[df_2025['county'] == 'Collin', 'Median_H_Inc'] = (df_2025.loc[df_2025['county'] == 'Collin', 'Median_H_Inc'] * CCAD_change_24).round().astype(int)

df_2025.loc[df_2025['county'] == 'Dallas', 'Median_H_Inc'] = (df_2025.loc[df_2025['county'] == 'Dallas', 'Median_H_Inc'] * DCAD_change_24).round().astype(int)

df_2025.loc[df_2025['county'] == 'Tarrant', 'Median_H_Inc'] = (df_2025.loc[df_2025['county'] == 'Tarrant', 'Median_H_Inc'] * TAD_change_24).round().astype(int)

df_2025.head()

,year,TRACT,GEO_ID,Median_H_Inc,TRACT_ID,county
1,2025,316.68,1400000US48085031668,279428,48085031668,Collin
2,2025,317.06,1400000US48085031706,279428,48085031706,Collin
3,2025,73.01,1400000US48113007301,283294,48113007301,Dallas
4,2025,76.04,1400000US48113007604,283294,48113007604,Dallas
5,2025,80,1400000US48113008000,283294,48113008000,Dallas


In [22]:
# To combine all of the census dataframes into a single df

df_CB = pd.concat([df_CB, df_2024, df_2025], axis=0, ignore_index=True)

df_CB.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11491 entries, 0 to 11490
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   year          11491 non-null  int64 
 1   TRACT         11491 non-null  object
 2   GEO_ID        11491 non-null  object
 3   Median_H_Inc  11491 non-null  object
 4   TRACT_ID      11491 non-null  object
 5   county        11491 non-null  object
dtypes: int64(1), object(5)
memory usage: 538.8+ KB


In [23]:
df_CB.to_csv('2019_2025_CB_HInc.csv')